In [58]:
import numpy as np
import datasets
from datasets import load_dataset
import pandas as pd
import altair as alt
import numpy as np
from helper_funks import show_me_more
alt.renderers.enable("mimetype")

RendererRegistry.enable('mimetype')

Initial data analysis

In [3]:
# Load our dataset
ds = load_dataset("trl-lib/tldr")
ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 116722
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 6447
    })
    test: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 6553
    })
})

In [3]:
# Looking at the training, test and validation data to examine them
training = pd.read_excel(open('subsets.xlsx', 'rb'), sheet_name='Train')
testing = pd.read_excel(open('subsets.xlsx', 'rb'), sheet_name='Test')
validation = pd.read_excel(open('subsets.xlsx', 'rb'), sheet_name='Validation')

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [ ]:
# showing the amount of posts in each subreddit (category) in the training set
bar = show_me_more(training, "Train data", 'bluepurple')
bar.display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [ ]:
# showing the amount of posts in each subreddit (category) in the test set
bar = show_me_more(testing, "Test data", 'greenblue')
bar.display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [ ]:
# showing the amount of posts in each subreddit (category) in the validation set
bar = show_me_more(validation, "Validation data", 'orangered', False)
bar.display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


Preliminary model evaluation and metrics analysis

In [59]:
#Import the TL;DR dataset.
from datasets import load_dataset
posts = load_dataset("trl-lib/tldr")

In [60]:
#Method for getting the subreddit from the prompt text.
def getSubredditFromPost(post):
    return  post['prompt'].split('\n')[0][11:];

#Get all subreddits and the amount of posts present within each subreddit.
counts = {}
subreddits = []
for post in posts['train']:
    subreddit = getSubredditFromPost(post)
    if subreddit not in subreddits:
        subreddits.append(subreddit)
    if(subreddit in counts):
        counts[subreddit] = counts[subreddit] + 1
    else:
        counts[subreddit] = 1

#To ensure equal representation of subreddits we evaluate the same amount of posts from each subreddit. 
#That means we will take the amount of posts in the subreddit that has fewest posts and evaluate this 
#amount of posts for each subreddit.
amount_of_posts_to_evaluate = min(counts.values())


In [61]:
import time
import evaluate
from helper_funks import NOIR

#Load the rouge and bleurt metric for evaluation.
rouge = evaluate.load("rouge")
bleurt = evaluate.load("bleurt", module_type="metric")
bertscore = evaluate.load("bertscore")
bleu = evaluate.load("bleu")

#Compute the NOIR, bleurt and rouge scores for the given pipeline.
def compute_scores(pipe, model_name):
  noirScores = {}
  bleurtScores = {}
  rougeScores = {}
  bertScores = {}
  bleuScores = {}

  promtsAndSummaries = {"prompt": [], model_name: []}
  for subreddit in subreddits:
    start = time.time()

    noirScores[subreddit] = []

    #Get the prompts and the labels of the given subreddit.
    sub_dataset = posts.filter(lambda post : getSubredditFromPost(post) == subreddit)['train'].select(range(amount_of_posts_to_evaluate))
    sub_dataset_prompts = [p['prompt'] for p in sub_dataset]
    labels = [p['completion'] for p in sub_dataset]
    
    #Generate the summaries from the given pipeline.
    generatedTexts = [text['summary_text'] for text in pipe(sub_dataset_prompts)]
    #Compute the NOIR score for each summary
    for prompt, generatedText in zip(sub_dataset_prompts, generatedTexts):
      promtsAndSummaries["prompt"].append(prompt)
      promtsAndSummaries[model_name].append(generatedText)
      
      noirScore = NOIR(prompt, generatedText)
      noirScores[subreddit].append(noirScore)
    
    #Compute the BLEURT score for each summary
    bleurtScore = bleurt.compute(predictions=generatedTexts, references=labels)
    bleurtScores[subreddit] = bleurtScore['scores']

    #Compute the average ROUGE-1, ROUGE-2, ROUGE-L and ROUGE-Lsum for the generated summaries.
    rougeScore = rouge.compute(predictions=generatedTexts, references=labels)
    rougeScores[subreddit] = rougeScore

    #Compute the BERT Score for each summary
    bertScore = bertscore.compute(predictions=generatedTexts, references=labels, lang='en')
    bertScores[subreddit] = bertScore

    #Compute the BLEU Score for each summary
    bleuScore = bleu.compute(predictions=generatedTexts, references=labels)
    bleuScores[subreddit] = bleuScore

    print(subreddit + " time: " + str(time.time()-start))

  return noirScores, bleurtScores, rougeScores, bertScores, bleuScores, promtsAndSummaries


INFO:tensorflow:Reading checkpoint C:\Users\Tobias\.cache\huggingface\metrics\bleurt\default\downloads\extracted\30ddb91ee5d94d9d15afa72de293cfbd181c7d7b153f001020ed5092a3320c9d\bleurt-base-128.


INFO:tensorflow:Reading checkpoint C:\Users\Tobias\.cache\huggingface\metrics\bleurt\default\downloads\extracted\30ddb91ee5d94d9d15afa72de293cfbd181c7d7b153f001020ed5092a3320c9d\bleurt-base-128.


INFO:tensorflow:Config file found, reading.


INFO:tensorflow:Config file found, reading.


INFO:tensorflow:Will load checkpoint bert_custom


...

INFO:tensorflow:Creating Eager Mode predictor.


INFO:tensorflow:Loading model.


INFO:tensorflow:Loading model.


INFO:tensorflow:BLEURT initialized.


INFO:tensorflow:BLEURT initialized.


In [62]:
import pandas as pd
def storePromptsAndSummaries(promtsAndSummaries):
    try:
        original_df = pd.read_csv("./summaries.csv")
    except:
        original_df = pd.DataFrame({"prompt": []})
    new_df = pd.DataFrame({k: v for k, v in promtsAndSummaries.items()})
    final_df = pd.merge(original_df, new_df, on='prompt', how='outer')
    final_df.to_csv('./summaries.csv', index=False)
    

In [63]:
#Evaluate the bart base model from ainize
from transformers import pipeline
ainize_noirScores, ainize_bleurtScores, ainize_rougeScores, ainize_bertScores, ainize_bleuScores, promtsAndSummaries = compute_scores(pipeline("summarization", model="ainize/bart-base-cnn", device=0), "ainize/bart-base-cnn")
storePromptsAndSummaries(promtsAndSummaries)


Device set to use cuda:0
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


r/relationships time: 162.5468044281006


Your max_length is set to 128, but your input_length is only 73. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=36)
Your max_length is set to 128, but your input_length is only 121. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=60)


r/loseit time: 141.21533060073853


Your max_length is set to 128, but your input_length is only 93. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=46)


...

r/GetMotivated time: 136.47758889198303


Your max_length is set to 128, but your input_length is only 86. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=43)


r/Parenting time: 168.11462712287903
r/weddingplanning time: 150.19961071014404
r/college time: 132.20771288871765
r/jobs time: 135.89620542526245
r/Dogtraining time: 160.77237105369568
r/Pets time: 157.7171115875244


Your max_length is set to 128, but your input_length is only 107. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=53)


r/Cooking time: 123.53223538398743
r/askwomenadvice time: 153.02450489997864
r/AskDocs time: 157.00573992729187
r/travel time: 129.3330521583557
r/books time: 125.23175621032715


In [64]:
#Save the scores to a csv and a json file.
import pandas as pd
from transformers import pipeline
df = pd.DataFrame(ainize_noirScores)
df.to_json('ainize_noirScores.json', index=False)
df.to_csv('ainize_noirScores.csv', index=False)
df = pd.DataFrame(ainize_bleurtScores)
df.to_json('ainize_bleurtScores.json', index=False)
df.to_csv('ainize_bleurtScores.csv', index=False)
df = pd.DataFrame(ainize_rougeScores)
df.to_json('ainize_rougeScores.json', index=False)
df.to_csv('ainize_rougeScores.csv', index=False)
df = pd.DataFrame(ainize_bertScores)
df.to_json('ainize_bertScores.json', index=False)
df.to_csv('ainize_bertScores.csv', index=False)
df = pd.DataFrame(ainize_bleuScores)
df.to_json('ainize_bleuScores.json', index=False)
df.to_csv('ainize_bleuScores.csv', index=False)

In [65]:
#Evaluate the t5-small model from falconsai.
from transformers import pipeline
falconsai_noirScores, falconsai_bleurtScores, falconsai_rougeScores, falconsai_bertScores, falconsai_bleuScores, promtsAndSummaries = compute_scores(pipeline("summarization", model="Falconsai/text_summarization", device=0), "Falconsai/text_summarization")
storePromptsAndSummaries(promtsAndSummaries)

Device set to use cuda:0
Token indices sequence length is longer than the specified maximum sequence length for this model (519 > 512). Running this sequence through the model will result in indexing errors
Your max_length is set to 200, but your input_length is only 196. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=98)


r/relationships time: 140.20046877861023


Your max_length is set to 200, but your input_length is only 177. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=88)
Your max_length is set to 200, but your input_length is only 78. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=39)
Your max_length is set to 200, but your input_length is only 199. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Your max_length is set to 200, but your input_length is only 167. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=83)
Y

r/loseit time: 140.5603506565094


Your max_length is set to 200, but your input_length is only 197. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=98)
Your max_length is set to 200, but your input_length is only 173. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=86)
Your max_length is set to 200, but your input_length is only 96. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=48)
Your max_length is set to 200, but your input_length is only 199. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Y

...

r/AskDocs time: 158.74709963798523


Your max_length is set to 200, but your input_length is only 189. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=94)
Your max_length is set to 200, but your input_length is only 179. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=89)
Your max_length is set to 200, but your input_length is only 198. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Your max_length is set to 200, but your input_length is only 195. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)


r/travel time: 139.60097885131836


Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 171. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=85)


r/books time: 146.04990196228027


In [66]:
#Save the scores to a csv and a json file.
import pandas as pd
df = pd.DataFrame(falconsai_noirScores)
df.to_json('falconsai_noirScores.json', index=False)
df.to_csv('falconsai_noirScores.csv', index=False)
df = pd.DataFrame(falconsai_bleurtScores)
df.to_json('falconsai_bleurtScores.json', index=False)
df.to_csv('falconsai_bleurtScores.csv', index=False)
df = pd.DataFrame(falconsai_rougeScores)
df.to_json('falconsai_rougeScores.json', index=False)
df.to_csv('falconsai_rougeScores.csv', index=False)
df = pd.DataFrame(falconsai_bertScores)
df.to_json('falconsai_bertScores.json', index=False)
df.to_csv('falconsai_bertScores.csv', index=False)
df = pd.DataFrame(falconsai_bleuScores)
df.to_json('falconsai_bleuScores.json', index=False)
df.to_csv('falconsai_bleuScores.csv', index=False)

In [67]:
#Evaluate the pegasus large model from google
from transformers import pipeline
pipe = pipeline("summarization", model="google/pegasus-large", device=0)
pegasus_noirScores, pegasus_bleurtScores, pegasus_rougeScores, pegasus_bertScores, pegasus_bleuScores, promtsAndSummaries = compute_scores(pipe, "google/pegasus-large")
storePromptsAndSummaries(promtsAndSummaries)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0
Your max_length is set to 256, but your input_length is only 186. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=93)
Your max_length is set to 256, but your input_length is only 220. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=110)
Your max_length is set to 256, but your input_length is only 177. Since this is a summarization task, where outputs shorter than the inp

r/relationships time: 984.2194845676422


Your max_length is set to 256, but your input_length is only 191. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=95)
Your max_length is set to 256, but your input_length is only 158. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)
Your max_length is set to 256, but your input_length is only 198. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Your max_length is set to 256, but your input_length is only 221. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=110)

r/loseit time: 897.1494872570038


Your max_length is set to 256, but your input_length is only 166. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=83)
Your max_length is set to 256, but your input_length is only 150. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=75)
Your max_length is set to 256, but your input_length is only 243. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=121)
Your max_length is set to 256, but your input_length is only 82. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=41)


...

r/AskDocs time: 1016.8745849132538


Your max_length is set to 256, but your input_length is only 239. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=119)
Your max_length is set to 256, but your input_length is only 186. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=93)
Your max_length is set to 256, but your input_length is only 215. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=107)
Your max_length is set to 256, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97

r/travel time: 886.2831847667694


Your max_length is set to 256, but your input_length is only 218. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=109)
Your max_length is set to 256, but your input_length is only 204. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=102)
Your max_length is set to 256, but your input_length is only 224. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=112)
Your max_length is set to 256, but your input_length is only 193. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=9

r/books time: 870.8760530948639


In [68]:
#Save the scores to a csv and a json file.
import pandas as pd
df = pd.DataFrame(pegasus_noirScores)
df.to_json('pegasus_noirScores.json', index=False)
df.to_csv('pegasus_noirScores.csv', index=False)
df = pd.DataFrame(pegasus_bleurtScores)
df.to_json('pegasus_bleurtScores.json', index=False)
df.to_csv('pegasus_bleurtScores.csv', index=False)
df = pd.DataFrame(pegasus_rougeScores)
df.to_json('pegasus_rougeScores.json', index=False)
df.to_csv('pegasus_rougeScores.csv', index=False)
df = pd.DataFrame(pegasus_bertScores)
df.to_json('pegasus_bertScores.json', index=False)
df.to_csv('pegasus_bertScores.csv', index=False)
df = pd.DataFrame(pegasus_bleuScores)
df.to_json('pegasus_bleuScores.json', index=False)
df.to_csv('pegasus_bleuScores.csv', index=False)

In [69]:
#Evaluate the t5 large model from google
from transformers import pipeline
pipe = pipeline("summarization", model="google-t5/t5-large", device=0)
google_t5_noirScores, google_t5_bleurtScores, google_t5_rougeScores, google_t5_bertScores, google_t5_bleuScores, promtsAndSummaries = compute_scores(pipe, "google-t5/t5-large")
storePromptsAndSummaries(promtsAndSummaries)

Device set to use cuda:0
Your max_length is set to 200, but your input_length is only 196. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=98)


r/relationships time: 387.3633482456207


Your max_length is set to 200, but your input_length is only 177. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=88)
Your max_length is set to 200, but your input_length is only 78. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=39)
Your max_length is set to 200, but your input_length is only 199. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Your max_length is set to 200, but your input_length is only 167. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=83)
Y

r/loseit time: 379.7075719833374


Your max_length is set to 200, but your input_length is only 197. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=98)
Your max_length is set to 200, but your input_length is only 173. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=86)
Your max_length is set to 200, but your input_length is only 96. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=48)
Your max_length is set to 200, but your input_length is only 199. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Y

...

r/AskDocs time: 405.676144361496


Your max_length is set to 200, but your input_length is only 189. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=94)
Your max_length is set to 200, but your input_length is only 179. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=89)
Your max_length is set to 200, but your input_length is only 198. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=99)
Your max_length is set to 200, but your input_length is only 195. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)


r/travel time: 358.3067092895508


Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 194. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=97)
Your max_length is set to 200, but your input_length is only 171. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=85)


r/books time: 357.7131052017212


In [70]:
#Save the scores to a csv and a json file.
import pandas as pd
df = pd.DataFrame(google_t5_noirScores)
df.to_json('google_t5_noirScores.json', index=False)
df.to_csv('google_t5_noirScores.csv', index=False)
df = pd.DataFrame(google_t5_bleurtScores)
df.to_json('google_t5_bleurtScores.json', index=False)
df.to_csv('google_t5_bleurtScores.csv', index=False)
df = pd.DataFrame(google_t5_rougeScores)
df.to_json('google_t5_rougeScores.json', index=False)
df.to_csv('google_t5_rougeScores.csv', index=False)
df = pd.DataFrame(google_t5_bertScores)
df.to_json('google_t5_bertScores.json', index=False)
df.to_csv('google_t5_bertScores.csv', index=False)
df = pd.DataFrame(google_t5_bleuScores)
df.to_json('google_t5_bleuScores.json', index=False)
df.to_csv('google_t5_bleuScores.csv', index=False)

In [71]:
#Evaluate the bart large model from google
from transformers import pipeline
pipe = pipeline("summarization", model="facebook/bart-large-cnn", device=0)
facebook_bart_noirScores, facebook_bart_bleurtScores, facebook_bart_rougeScores, facebook_bart_bertScores, facebook_bart_bleuScores, promtsAndSummaries = compute_scores(pipe, "facebook/bart-large-cnn")
storePromptsAndSummaries(promtsAndSummaries)

Device set to use cuda:0


r/relationships time: 352.9230914115906


Your max_length is set to 142, but your input_length is only 72. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=36)
Your max_length is set to 142, but your input_length is only 120. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=60)


r/loseit time: 340.99231576919556


Your max_length is set to 142, but your input_length is only 92. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=46)


...

r/Parenting time: 341.1926808357239
r/weddingplanning time: 348.22579050064087


Your max_length is set to 142, but your input_length is only 134. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=67)


r/college time: 312.2671456336975
r/jobs time: 335.6937744617462
r/Dogtraining time: 326.3661241531372
r/Pets time: 328.491557598114


Your max_length is set to 142, but your input_length is only 106. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=53)


r/Cooking time: 283.5791745185852
r/askwomenadvice time: 332.05876779556274
r/AskDocs time: 339.91633439064026
r/travel time: 290.1241545677185
r/books time: 307.89886379241943


In [72]:
#Save the scores to a csv and a json file.
import pandas as pd
df = pd.DataFrame(facebook_bart_noirScores)
df.to_json('facebook_bart_noirScores.json', index=False)
df.to_csv('facebook_bart_noirScores.csv', index=False)
df = pd.DataFrame(facebook_bart_bleurtScores)
df.to_json('facebook_bart_bleurtScores.json', index=False)
df.to_csv('facebook_bart_bleurtScores.csv', index=False)
df = pd.DataFrame(facebook_bart_rougeScores)
df.to_json('facebook_bart_rougeScores.json', index=False)
df.to_csv('facebook_bart_rougeScores.csv', index=False)
df = pd.DataFrame(facebook_bart_bertScores)
df.to_json('facebook_bart_bertScores.json', index=False)
df.to_csv('facebook_bart_bertScores.csv', index=False)
df = pd.DataFrame(facebook_bart_bleuScores)
df.to_json('facebook_bart_bleuScores.json', index=False)
df.to_csv('facebook_bart_bleuScores.csv', index=False)

Plotting the metrics

In [12]:
# Bart-Base: Ainize
# getting the data as dataframe and reformating them so we can plot them
df_ainize_bleurt = pd.read_csv('evals/ainize_bleurtScores.csv').melt(var_name='column')
df_ainize_noir = pd.read_csv('evals/ainize_noirScores.csv').melt(var_name='column')
# change names of the dataframes for easy plotting
source_1_1 = pd.DataFrame(df_ainize_bleurt)
source_1_2 = pd.DataFrame(df_ainize_noir)

In [13]:
# T5 small - falconsai
# getting the data as dataframe and reformating them so we can plot them
df_falconsai_bleurt = pd.read_csv('evals/falconsai_bleurtScores.csv').melt(var_name='column')
df_falconsai_noir = pd.read_csv('evals/falconsai_noirScores.csv').melt(var_name='column')
# change names of the dataframes for easy plotting
source_2_1 = pd.DataFrame(df_falconsai_bleurt)
source_2_2 = pd.DataFrame(df_falconsai_noir)

In [14]:
# Pegasus
# getting the data as dataframe and reformating them so we can plot them
df_pegasus_bleurt = pd.read_csv('evals/pegasus_bleurtScores.csv').melt(var_name='column')
df_pegasus_noir = pd.read_csv('evals/pegasus_noirScores.csv').melt(var_name='column')
# change names of the dataframes for easy plotting
source_3_1 = pd.DataFrame(df_pegasus_bleurt)
source_3_2 = pd.DataFrame(df_pegasus_noir)

In [73]:
# Plot of the three models with Bleurt metrics
one = alt.Chart(source_1_1, title="ainize_bleurt").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)
two = alt.Chart(source_2_1, title="falconai_bleurt").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)
three = alt.Chart(source_3_1, title="pegasus_bleurt").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)

(one & two & three).properties(width=800).interactive()
(one & two & three).display()

NameError: name 'source_1_1' is not defined

In [16]:
# Plot of the three models with Noir metrics
one = alt.Chart(source_1_2, title="ainize_noir").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)
two = alt.Chart(source_2_2, title="falconai_noir").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)
three = alt.Chart(source_3_2, title="pegasus_noir").mark_boxplot().encode(
alt.X('column').axis(labels=False),
y='value',
color='column'
)

(one & two & three).properties(width=800).interactive()
(one & two & three).display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [17]:
# Rouge metrics
# Rouge metrics had a different format than Bleurt and Noir, so we needed to plot the results in a different way, we chose line charts
df_ainize_rouge = pd.read_csv('evals/ainize_rougeScores.csv')
df_falconsai_rouge = pd.read_csv('evals/falconsai_rougeScores.csv')
df_pegases_rouge = pd.read_csv('evals/pegases_rougeScores.csv')

# Melt the dataframes to long format
df_ain_rouge = df_ainize_rouge.reset_index().melt(id_vars='index', var_name='Subreddit', value_name='Value')
df_falc_rouge = df_falconsai_rouge.reset_index().melt(id_vars='index', var_name='Subreddit', value_name='Value')
df_peg_rouge = df_pegases_rouge.reset_index().melt(id_vars='index', var_name='Subreddit', value_name='Value')

In [18]:
# the line charts for the three models
one = (
    alt.Chart(df_ain_rouge, title='ainize_Rouge')
    .mark_line()
    .encode(
        x='Subreddit:N',
        y='Value:Q',
        color='index:N'
    )
)
two = (
    alt.Chart(df_falc_rouge, title='Falconai_Rouge')
    .mark_line()
    .encode(
        alt.X('Subreddit:N').axis(labels=False),
        y='Value:Q',
        color='index:N'
    )
)
three = (
    alt.Chart(df_peg_rouge, title='Pegasus_Rouge')
    .mark_line()
    .encode(
        alt.X('Subreddit:N').axis(labels=False),
        y='Value:Q',
        color='index:N'
    )
)

(one & two & three).properties(width=800).interactive()
(one & two & three).display()

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting
